# Chapter 33 Coding Exam — Rigorous Multi-Seed Benchmarking, Exact McNemar & Cluster-CV Ablation (100 Points)

**Book Reference:** Chapter 33 (Opening Part VII: Research Capstone), Section 33.7 (Coding Exam)  
**Suggested Time:** 50–60 minutes (Open Book, Closed Solution)

| Task | Experimental Rigor Deliverable | Points |
| :--- | :--- | :---: |
| **Task 33.1** | `multi_seed_cv_benchmark(X, y, model_factory, seeds, cv_splits=4)` (Leak-free multi-seed CV summary) | 30 |
| **Task 33.2** | `mcnemar_exact_comparison(y_true, pred_a, pred_b)` (Exact binomial McNemar paired classifier test) | 35 |
| **Task 33.3** | `mch_ablation_study_report(mch_df, seed=42)` (Cluster-aware `GroupKFold` feature-group ablation on `child_stunted`) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 33 Exam Grader
!pip install -q sharukh-tutor
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GroupKFold,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

tutor.activate(chapter="ch33", notebook="coding_exam")


---
## Exam Task 33.1 (30 Points): `multi_seed_cv_benchmark(X, y, model_factory, seeds, cv_splits=4)`

Implement `multi_seed_cv_benchmark(X, y, model_factory, seeds, cv_splits=4)`:
1. For every seed `s` in `seeds`, build `StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=s)` and a fresh leak-free `Pipeline([("scaler", StandardScaler()), ("model", model_factory())])`.
2. Compute the mean `roc_auc` via `cross_val_score(..., scoring="roc_auc")` for each seed.
3. Return `{"n_seeds": int, "mean_auc": round4, "std_auc": round4, "min_auc": round4, "max_auc": round4, "seed_scores": list_round4}`.

In [ ]:
def multi_seed_cv_benchmark(
    X, y, model_factory, seeds, cv_splits=4
):
    """Run leak-free Stratified CV across multiple random seeds."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    raw = []
    for s in seeds:
        skf = StratifiedKFold(
            n_splits=cv_splits, shuffle=True, random_state=s
        )
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model_factory()),
        ])
        sc = cross_val_score(pipe, X, y, cv=skf, scoring="roc_auc")
        raw.append(float(sc.mean()))
    return {
        "n_seeds": int(len(seeds)),
        "mean_auc": round(float(np.mean(raw)), 4),
        "std_auc": round(float(np.std(raw)), 4),
        "min_auc": round(float(np.min(raw)), 4),
        "max_auc": round(float(np.max(raw)), 4),
        "seed_scores": [round(float(v), 4) for v in raw],
    }
    ### END SOLUTION


---
## Exam Task 33.2 (35 Points): `mcnemar_exact_comparison(y_true, pred_a, pred_b)`

Implement `mcnemar_exact_comparison(y_true, pred_a, pred_b)`:
1. Count `b` = number of examples where `pred_a` is correct AND `pred_b` is wrong.
2. Count `c` = number of examples where `pred_a` is wrong AND `pred_b` is correct.
3. Compute the exact two-sided binomial p-value via `scipy.stats.binomtest(min(b, c), b + c, 0.5).pvalue` (or `1.0` if `b + c == 0`).
4. Return `{"b_only_a_correct": int, "c_only_b_correct": int, "p_value": round4, "significant_05": bool}`.

In [ ]:
def mcnemar_exact_comparison(y_true, pred_a, pred_b):
    """Run exact two-sided McNemar binomial test on paired preds."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    yt = np.asarray(y_true)
    pa = np.asarray(pred_a)
    pb = np.asarray(pred_b)
    a_ok = pa == yt
    b_ok = pb == yt
    b = int(np.sum(a_ok & ~b_ok))
    c = int(np.sum(~a_ok & b_ok))
    p_val = (
        1.0 if (b + c) == 0
        else float(binomtest(min(b, c), b + c, 0.5).pvalue)
    )
    return {
        "b_only_a_correct": b,
        "c_only_b_correct": c,
        "p_value": round(p_val, 4),
        "significant_05": bool(p_val < 0.05),
    }
    ### END SOLUTION


---
## Exam Task 33.3 (35 Points): `mch_ablation_study_report(mch_df, seed=42)`

Without mutating `mch_df`, run a cluster-aware feature-group ablation study predicting `child_stunted`:
1. Copy `mch_df`, standardize `residence` casing via `.astype(str).str.strip().str.title()`, derive `is_urban = (work["residence"] == "Urban").astype(int)`, map `wealth_quintile` (`Poorest: 1 .. Richest: 5`) to `wealth_rank`, replace `anc_visits == 999` with `np.nan`, and drop incomplete rows across the 5 features + `child_stunted` + `cluster_id`.
2. Define feature groups: `demographics=["maternal_age_years", "is_urban"]`, `care_access=["anc_visits"]`, `socioeconomic=["wealth_rank"]`, `nutrition=["maternal_hb_g_dl"]`.
3. Score `Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=500))])` with `GroupKFold(n_splits=5)` grouped by `cluster_id` (`scoring="roc_auc"`) on `"full_model"` and each `"drop_<group>"` subset.
4. Build `ablation_table` with columns `["ablation", "mean_auc", "std_auc", "delta_vs_full"]` (rounded to 4 decimals), identify `most_critical_group` (bare group name with most negative `delta_vs_full`), and return `{"full_auc": round4, "ablation_table": DataFrame, "most_critical_group": str}`.

In [ ]:
def mch_ablation_study_report(mch_df, seed=42):
    """Run cluster-aware GroupKFold ablation study on MCH stunting."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    work = mch_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    wealth_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_rank"] = (
        work["wealth_quintile"].astype(str).str.strip()
        .str.title().map(wealth_map)
    )
    feature_groups = {
        "demographics": ["maternal_age_years", "is_urban"],
        "care_access": ["anc_visits"],
        "socioeconomic": ["wealth_rank"],
        "nutrition": ["maternal_hb_g_dl"],
    }
    all_cols = [c for cols in feature_groups.values() for c in cols]
    clean = (
        work[all_cols + ["child_stunted", "cluster_id"]]
        .dropna()
        .reset_index(drop=True)
    )
    X = clean[all_cols]
    y = clean["child_stunted"].astype(int)
    clusters = clean["cluster_id"]

    gkf = GroupKFold(n_splits=5)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500)),
    ])
    full_sc = cross_val_score(
        pipe, X, y, groups=clusters, cv=gkf, scoring="roc_auc"
    )
    full_auc = float(full_sc.mean())
    rows = [{
        "ablation": "full_model",
        "mean_auc": round(full_auc, 4),
        "std_auc": round(float(full_sc.std()), 4),
        "delta_vs_full": 0.0,
    }]
    for gname, cols in feature_groups.items():
        sc = cross_val_score(
            pipe, X.drop(columns=cols), y,
            groups=clusters, cv=gkf, scoring="roc_auc",
        )
        m = float(sc.mean())
        rows.append({
            "ablation": f"drop_{gname}",
            "mean_auc": round(m, 4),
            "std_auc": round(float(sc.std()), 4),
            "delta_vs_full": round(m - full_auc, 4),
        })
    tbl = pd.DataFrame(rows)
    drops = tbl[tbl["ablation"] != "full_model"]
    critical = str(
        drops.loc[drops["delta_vs_full"].idxmin(), "ablation"]
    ).removeprefix("drop_")
    return {
        "full_auc": round(full_auc, 4),
        "ablation_table": tbl,
        "most_critical_group": critical,
    }
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch33_exam_01": multi_seed_cv_benchmark,
    "ch33_exam_02": mcnemar_exact_comparison,
    "ch33_exam_03": mch_ablation_study_report,
}

tutor.check_exam("ch33", exam_submissions)


---
## Chapter 33 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 33.1 (Fresh Estimators Per Seed):** Calling `model_factory()` once outside the loop reuses the same estimator instance across seeds. Always instantiate `Pipeline([("scaler", StandardScaler()), ("model", model_factory())])` inside the loop so every seed starts from a completely clean slate.
- **Task 33.2 (Zero-Discordance Guard in Exact McNemar):** When two classifiers agree on every single test case (`b + c == 0`), calling `scipy.stats.binomtest(0, 0, 0.5)` raises `ValueError: n must be a positive integer`. Always return `p_value = 1.0` whenever `b + c == 0`.
- **Task 33.3 (Cluster-Aware Ablation & Prefix Stripping):** Dropping `socioeconomic` (`wealth_rank`) slashes 5-fold `GroupKFold` ROC-AUC from `0.7127` to `0.6530` (`delta = -0.0597`), proving socioeconomic status is the dominant predictor of child stunting across unseen survey clusters. Remember to strip the `"drop_"` prefix when reporting `most_critical_group`.